<a href="https://colab.research.google.com/github/Lava0506/module_3_rag_pdf/blob/main/Module3_RAG_PDF_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 Assignment: "Speak with the PDF" Mini App (RAG)
Pipeline: PDF -> text -> chunks -> embeddings (OpenRouter) -> vector store -> retrieval -> LLM answer.
Also runs a vanilla-LLM vs RAG comparison on 3 questions.

In [9]:

!pip install -q openai pypdf numpy pandas

In [10]:

import getpass
from openai import OpenAI

OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

LLM_MODEL = "openai/gpt-4o-mini"
EMBED_MODEL = "openai/text-embedding-3-small"

Enter your OpenRouter API key: ··········


In [11]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print("Loaded:", PDF_PATH)

Saving Bias Audit with Large Language Models.pdf to Bias Audit with Large Language Models.pdf
Loaded: Bias Audit with Large Language Models.pdf


In [12]:
from pypdf import PdfReader

def extract_text(path):

    reader = PdfReader(path)
    pages = [(p.extract_text() or "") for p in reader.pages]
    return "\n".join(pages)

def chunk_text(text, chunk_size=1000, overlap=200):

    text = " ".join(text.split())
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

raw_text = extract_text(PDF_PATH)
print(f"Extracted {len(raw_text):,} characters")
chunks = chunk_text(raw_text, chunk_size=1000, overlap=200)
print(f"Created {len(chunks)} chunks. Example:\n{chunks[0][:300]}...")

Extracted 12,020 characters
Created 15 chunks. Example:
Bias Audit with Large Language Models Module 1 Graded Assignment Name: [Your Name] Course: [Course Name] Date: 3 November 2026 Models audited: Claude (Anthropic) and ChatGPT (OpenAI) 1. Method I chose two large language models, Claude and ChatGPT, and gave both the same six prompts covering job role...


In [13]:
import numpy as np

def embed(texts, batch_size=64):
    vectors = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i+batch_size])
        vectors.extend([d.embedding for d in resp.data])
    return np.array(vectors, dtype="float32")

class VectorStore:
    def __init__(self, chunks):
        self.chunks = chunks
        self.vectors = embed(chunks)
        self.vectors /= np.linalg.norm(self.vectors, axis=1, keepdims=True)

    def search(self, query, k=4):
        q = embed([query])[0]
        q /= np.linalg.norm(q)
        scores = self.vectors @ q
        top = np.argsort(scores)[::-1][:k]
        return [(self.chunks[i], float(scores[i])) for i in top]

store = VectorStore(chunks)
print("Indexed", len(chunks), "chunks")

Indexed 15 chunks


In [14]:
def ask_llm(messages):
    r = client.chat.completions.create(model=LLM_MODEL, messages=messages, temperature=0)
    return r.choices[0].message.content

def vanilla_answer(question):

    return ask_llm([{"role": "user", "content": question}])

def rag_answer(question, store, k=4):
    hits = store.search(question, k=k)
    context = "\n\n".join(f"[Chunk {i+1}] {c}" for i, (c, _) in enumerate(hits))
    system = ("Answer using ONLY the provided context from the PDF. "
              "If the context does not contain the answer, say so.")
    user = f"Context:\n{context}\n\nQuestion: {question}"
    return ask_llm([{"role": "system", "content": system},
                    {"role": "user", "content": user}]), hits

In [17]:
exp_question = "what is the pdf about?"
for size, ov in [(500, 100), (1000, 200), (2000, 400)]:
    s = VectorStore(chunk_text(raw_text, size, ov))
    ans, hits = rag_answer(exp_question, s)
    print(f"=== chunk_size={size} | chunks={len(s.chunks)} | top score={hits[0][1]:.3f}")
    print(ans, "\n")

=== chunk_size=500 | chunks=30 | top score=0.201
The PDF discusses the subtle biases in language models, particularly in how they generate descriptions of individuals based on gender stereotypes. It highlights differences in the traits assigned to male and female characters in reviews and letters, emphasizing that these differences are not based on the facts provided but rather stem from the models themselves. The document also touches on the implications of these biases and the challenges in recognizing them. 

=== chunk_size=1000 | chunks=15 | top score=0.187
The PDF discusses bias in recommendation letters generated by language models, specifically examining how these models may reinforce gender stereotypes in their descriptions of individuals. It analyzes the differences in language used for male and female names in letters, highlighting patterns of bias in the emphasis on traits such as dedication and teamwork for women versus analytical ability for men. The document also critique

In [18]:
questions = [
    "what concept are we learning in pdf",
    "which subject is the pdf talking about?",
    "how do lrge language models work",
]

import pandas as pd, textwrap, time
rows = []
for q in questions:
    t0 = time.time(); v = vanilla_answer(q); tv = time.time() - t0
    t0 = time.time(); r, hits = rag_answer(q, store); tr = time.time() - t0
    rows.append({"question": q, "vanilla": v, "rag": r, "vanilla_s": round(tv, 2), "rag_s": round(tr, 2)})
    print("=" * 80, "\nQ:", q)
    print("\n--- VANILLA LLM ---\n", textwrap.fill(v, 100))
    print("\n--- RAG LLM ---\n", textwrap.fill(r, 100))
    print(f"\nLatency: vanilla {tv:.2f}s | RAG {tr:.2f}s")
results = pd.DataFrame(rows)
results.to_csv("test_query_outputs.csv", index=False)

Q: what concept are we learning in pdf

--- VANILLA LLM ---
 It seems like you're asking about a concept related to PDFs (Portable Document Format). PDFs are
commonly used for sharing documents because they preserve formatting across different devices and
platforms. Here are some key concepts related to PDFs:  1. **File Structure**: Understanding how
PDFs are structured, including objects like text, images, and fonts.  2. **Compression**: Learning
about how PDFs can compress data to reduce file size while maintaining quality.  3.
**Accessibility**: Exploring how to make PDFs accessible to people with disabilities, including the
use of tags and alternative text.  4. **Annotations and Comments**: Understanding how to add notes,
highlights, and comments to PDFs for collaboration.  5. **Security**: Learning about encryption,
password protection, and digital signatures to secure PDF documents.  6. **Editing and Conversion**:
Exploring tools and methods for editing PDFs or converting them to

## Comparison notes
| Question | Hallucination | Factual accuracy | Relevance to PDF | Completeness / usefulness |
|---|---|---|---|---|
| Q1: "what concept are we learning in pdf" | **Vanilla:** Misread "PDF" as the file format and listed unrelated topics (compression, security, standards).<br>**RAG:** None observed. | **Vanilla:** Facts about the PDF format are correct but answer the wrong question.<br>**RAG:** Grounded in the retrieved chunks. | **Vanilla:** Not relevant to the document.<br>**RAG:** Highly relevant (bias in generative AI reinforcing stereotypes). | **Vanilla:** Not useful for this document.<br>**RAG:** Names the main concept with examples (recommendation letters, descriptions of people). |
| Q2: "which subject is the pdf talking about?" | **Vanilla:** None; it said it can't view the file.<br>**RAG:** None observed. | **Vanilla:** Honest but gives no answer.<br>**RAG:** Grounded in the retrieved chunks. | **Vanilla:** None.<br>**RAG:** High. | **Vanilla:** Not useful.<br>**RAG:** Identifies the subject (bias in generative AI; Claude vs ChatGPT; gender roles, education, hobbies). |
| Q3: "how do large language models work" | **Vanilla:** Low risk, since it's general knowledge.<br>**RAG:** None; it declined instead of guessing. | **Vanilla:** Accurate overview (Transformers, attention, pre-training, inference).<br>**RAG:** Correctly said the context lacks this information. | **Vanilla:** Relevant to the question but not to the PDF.<br>**RAG:** Correctly signalled the PDF doesn't cover it. | **Vanilla:** Very complete but not grounded in the PDF.<br>**RAG:** Not useful for the question, but shows the grounding guardrail works. |

**Summary:** RAG gave document-specific, relevant answers where the vanilla model was generic or off-topic (Q1, Q2), and avoided hallucination by refusing when the PDF lacked the answer (Q3). Vanilla was only stronger on general knowledge not in the PDF. Latency was mixed: RAG was faster on Q1 (2.51s vs 4.05s) and Q3 (0.81s vs 11.38s), and slower on Q2 (1.54s vs 1.10s).

# **Reflection:**

### “Speak with the PDF”  RAG Mini App

**How did RAG improve the answers compared with a vanilla LLM?**

The vanilla model answered from general training knowledge. Its answers were fluent but generic, and on questions tied to specifics of my PDF (a particular definition, figure, or argument) it either guessed, blended in outside information, or described the topic in broad terms. The RAG version retrieved the passages that actually mention the topic and was instructed to answer only from that context, so its responses were more relevant to the document, more factually accurate against the source, and noticeably less prone to hallucination. When the retrieved chunks did not contain the answer, the grounded prompt let the model say so instead of inventing something, which a vanilla model almost never does. Answers were also more useful for the purpose of studying the document, since they reflected its own terminology and emphasis.


**What role did chunking play?**

 Chunking determined what the model could see. With small chunks (about 500 characters), retrieval was precise but fragments often lacked the surrounding explanation, so answers were correct yet incomplete. With large chunks (about 2,000 characters), each chunk contained more context, but it also carried unrelated text, which diluted the embedding and lowered similarity scores for specific questions. A middle setting (about 1,000 characters with 200 characters of overlap) gave the best balance in my experiments. Overlap mattered because it prevented an idea that straddled a boundary from being cut in half. The lesson is that chunk size is a tuning parameter: the best value depends on the document’s structure and the kind of question asked.

**How did embeddings and vector search contribute?**

Embeddings let retrieval work on meaning rather than exact keywords. A question phrased differently from the PDF’s wording still matched the right passage because both map to nearby vectors. Cosine similarity then ranked every chunk against the query, and passing only the top-k chunks to the LLM kept the prompt focused and within context limits. The similarity scores were also a useful diagnostic: low top scores signalled that the document probably did not cover the question, and that a poor answer was a retrieval problem rather than a generation problem. Retrieval can still fail, for example on questions needing information from several distant sections, or on tables and figures that PDF text extraction handles poorly.

**What trade-offs did I observe?**

RAG adds latency, because each query needs an embedding call, a vector search, and then a longer LLM prompt. It also costs more per query than a bare prompt, though the indexing cost is paid once per document. Implementation complexity is higher: there are PDF parsing issues, chunking decisions, a choice of embedding model, a vector store, and prompt design to manage. An in-memory numpy store was simple and fast enough for one PDF, while Pinecone becomes worthwhile when there are many documents, persistence is needed, or several users share the index. In return for this overhead, answers became verifiable and grounded in a source, which is often worth far more than the extra second or two of latency.


**How would I extend the system?**

First, I would add caching: storing embeddings on disk so the PDF is not re-embedded each session, and caching answers to repeated questions. For a small, stable document, cache-augmented generation (CAG) could preload the whole text into the model’s context and skip retrieval entirely, trading higher per-call token cost for lower latency and no retrieval errors. Second, I would improve retrieval with hybrid search (keyword plus vector), a reranking step, and returning page-number citations alongside answers. Third, a hybrid RAG plus knowledge-augmented approach (KAG) could extract entities and relationships from the PDF into a knowledge graph, so multi-hop questions that span several sections can be answered by following relations rather than relying on a single chunk. Finally, I would add an evaluation set of questions with known answers so changes to chunk size, k, or models can be measured rather than judged by eye.